In [1]:
import os

os.environ['http_proxy'] = 'http://localhost:3128'
os.environ['https_proxy'] = 'http://localhost:3128'
os.environ['HTTP_PROXY'] = 'http://localhost:3128'
os.environ['HTTPS_PROXY'] = 'http://localhost:3128'


In [2]:
import os

os.environ['HF_HOME'] = '/workspace/models/huggingface'
os.environ['HF_HUB_CACHE'] = '/workspace/models/huggingface/hub'
os.environ['TRANSFORMERS_CACHE'] = '/workspace/models/huggingface'

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="diffusers/controlnet-depth-sdxl-1.0-small",
    local_dir="/workspace/models/controlnet-depth-sdxl-small",
    local_dir_use_symlinks=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

'/workspace/models/controlnet-depth-sdxl-small'

In [3]:
import torch
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained(
    "/workspace/models/sdxl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")

prompt = """
side view of a driver inside a car, photographed from the front passenger seat angle,
both hands on the steering wheel, eyes on the road, safe driving,
natural daylight, high-resolution realistic DSLR photo
"""

image = pipe(
    prompt=prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    width=1024,	
    height=1024,
).images[0]

image.save("/workspace/generated-img/prompt-only/test_img_driver.png")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [4]:
from controlnet_aux import MidasDetector

depth_estimator = MidasDetector.from_pretrained("lllyasviel/Annotators")

dpt_hybrid-midas-501f0c75.pt:   0%|          | 0.00/493M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(


In [5]:
from PIL import Image
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel

base_model = "/workspace/models/sdxl-base-1.0/"
controlnet_model = "/workspace/models/controlnet-depth-sdxl-small/"

controlnet = ControlNetModel.from_pretrained(
    controlnet_model,
    torch_dtype=torch.float16,
    use_safetensors=True
    )

pipe= StableDiffusionXLControlNetPipeline.from_pretrained(
    base_model,
    controlnet=controlnet,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

real_img = Image.open("/workspace/imgs/train/c0/img_34.jpg").convert("RGB").resize((1024, 1024))
depth_map = depth_estimator(real_img)

depth_map.save("/workspace/generated-img/depth-sdxl/test_img_depth_map.png")

prompt = """
side view of a driver inside a car, photographed from the front passenger seat angle,
both hands on the steering wheel, eyes on the road, safe driving,
natural daylight, high-resolution realistic DSLR photo
"""

image = pipe(
    prompt=prompt,
    image=depth_map,
    num_inference_steps=30,
    guidance_scale=7.0,
    controlnet_conditioning_scale=0.6,
).images[0]

image.save("/workspace/generated-img/depth-sdxl/test_img_driver.png")


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/controlnet/pipeline_controlnet_sd_xl.py:928: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="thibaud/controlnet-openpose-sdxl-1.0",
    local_dir="/workspace/models/controlnet-openpose-sdxl",
    local_dir_use_symlinks=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

'/workspace/models/controlnet-openpose-sdxl'

In [1]:
from controlnet_aux import OpenposeDetector
from PIL import Image
import torch

pose_estimator = OpenposeDetector.from_pretrained("lllyasviel/Annotators")
real_img = Image.open("/workspace/imgs/train/c0/img_34.jpg").convert("RGB").resize((1024, 1024))
pose_map = pose_estimator(real_img) 
pose_map.save("/workspace/generated-img/pose-sdxl/test_img_pose_map.png")

ImportError: libGL.so.1: cannot open shared object file: No such file or directory

In [8]:
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel

base_model = "/workspace/models/sdxl-base-1.0"
controlnet_model = "/workspace/models/controlnet-openpose-sdxl"

controlnet = ControlNetModel.from_pretrained(
    controlnet_model,
    torch_dtype=torch.float16,
    use_safetensors=True
    )

pipe= StableDiffusionXLControlNetPipeline.from_pretrained(
    base_model,
    controlnet=controlnet,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

prompt = """
side view of a driver inside a car, photographed from the front passenger seat angle,
both hands on the steering wheel, eyes on the road, safe driving,
natural daylight, high-resolution realistic DSLR photo
"""

image = pipe(
    prompt=prompt,
    image=pose_map,
    num_inference_steps=30,
    guidance_scale=7.0,
    controlnet_conditioning_scale=0.6,
).images[0]

image.save("/workspace/generated-img/pose-sdxl/test_img_driver.png")

An error occurred while trying to fetch /workspace/models/controlnet-openpose-sdxl: Error no file named diffusion_pytorch_model.safetensors found in directory /workspace/models/controlnet-openpose-sdxl.


OSError: Error no file named diffusion_pytorch_model.safetensors found in directory /workspace/models/controlnet-openpose-sdxl.